# Análisis de Datos
## TP N 2
----
### Grupo N° 7
- Aviani, José
- Diaz, José Luis
- Silvera, Ricardo

---

## Introducción

Para este trabajo elegimos el el dataset Precios Claros – Base SEPA, perteneciente al “Sistema Electrónico de Publicidad de Precios Argentinos (SEPA)" (https://datos.gob.ar/), el cual reúne los precios de comercios minoristas (grandes establecimientos) de más de 70 mil productos en toda la Argentina. Particularmente para este trabajo, seleccionamos el set de datos del establecimiento **Carrefour** ya que era el de mayor tamaño, lo cual es deseable como entrada en un problema de aprendizaje de máquina.
A continuación realizamos la preparación de datos para IA y finalizamos con las conclusiones obtenidas del trabajo.

## Objetivo

A partir de la descripción del producto en particular vamos a generalizar un producto general. El objectivo es hacer la prepararción de datos que posteriormente nos permita seleccionar y entrenar un modelo para, dado el producto en general junto con el resto de los *features* relevantes, predecir el precio.

Este sería un problema de regresión supervisada donde la variable *target* es el precio. Se podría resolver con un árbol de decisión basado en ensamblado, como *Random Forest Regressor*, *Gradient Boosting Regressor* o *XGBoost/LightGBM*. De todos modos, esto está fuera del alcance del este TP.

---

In [ ]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
import re
import gc
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from unidecode import unidecode
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import LabelEncoder

---

### Carga del dataset

Como último paso de la parte 1 del TP hicimos un merge entre los diferentes datasets. Este será el dataset para la parte 2 del TP.

In [ ]:
# Leer el JSON
with open("./dataset/carrefour_dtypes.json", "r") as f:
    info = json.load(f)

dtypes_str = info["dtypes"]
categorical_cols = info["categoricals"]

# Convertir strings de tipo a los tipos correctos
def convertir_dtype(dtype_str):
    if dtype_str.startswith("int"): return "Int64"
    if dtype_str.startswith("float"): return "float"
    if dtype_str == "object": return "string"
    if dtype_str == "bool": return "boolean"
    return "string"

normal_dtypes = {
    col: convertir_dtype(dtype) for col, dtype in dtypes_str.items() if col not in categorical_cols
}

# Leer CSV
# Por limitaciones del tamaño maximo de un solo archivo que nos permite github separamos en varios archivos el dataset.
archivos = ['carrefour_part_01.csv.gz','carrefour_part_02.csv.gz','carrefour_part_03.csv.gz','carrefour_part_04.csv.gz']

df_con_nombre_de_columnas = pd.read_csv(f"./dataset/carrefour_part_00.csv.gz", compression="gzip", dtype=normal_dtypes, sep='|')
lista_de_dataframes = [df_con_nombre_de_columnas]

for archivo in archivos:
    df_temporal = pd.read_csv(f"./dataset/{archivo}", compression="gzip", dtype=normal_dtypes, sep='|', header=None, names=df_con_nombre_de_columnas.columns)
    lista_de_dataframes.append(df_temporal)

carrefour = pd.concat(lista_de_dataframes, ignore_index=True)

# Restaurar categoricas
for col in categorical_cols:
    carrefour[col] = carrefour[col].astype("category")

In [ ]:
carrefour.shape

In [ ]:
carrefour.info()

In [ ]:
pd.set_option('display.max_columns', None)
carrefour.head()

---

## Preparación de datos para IA

---

### Creación de nuevos features

El único feature identificado para crear es la generalización del nombre del producto.

Si bien es buena práctica hacer el split del dataset antes de aplicar las transformaciones para evitar riesgos de *data leakage*, en este caso, como no hay riesgo del mismo al agregar la nueva columna y para simplificar el proceso, lo hacemos antes.

Para generalizar el nombre del producto primero limpiamos el nombre, luego calculamos embeddings y los agrupamos en clusters (con KMeans). Luego, para cada clúster se extraen los tokens más frecuentes de sus descripciones y se asignan como etiqueta. Así, cada producto queda asociado a una categoría semántica sin marca. La nueva columna es "producto_base".

Definimos los métodos necesarios:

In [ ]:
# Limpieza del nombre sin marca
def clean_text_no_brand(desc: str, marca: str | None = None) -> str:
    def norm(s):
        s = (s or "").strip().lower()
        return unidecode(s)

    desc = norm(desc)
    marca = norm(marca) if marca else None

    # Tamaños/unidades/ruido
    desc = re.sub(r"\b\d+([.,]\d+)?\s?(g|kg|mg|ml|l|lt|lts|cc|cm3)\b", " ", desc)
    desc = re.sub(r"\bx\s?\d+\b", " ", desc)      # x6
    desc = re.sub(r"\bpack(s)?\b", " ", desc)
    desc = re.sub(r"\b\d+%|\d+\s?por\s?ciento\b", " ", desc)
    desc = re.sub(r"[^\w\s]", " ", desc)

    # Quitar marca y tokens de la marca
    if marca and len(marca) > 1:
        desc = re.sub(rf"\b{re.escape(marca)}\b", " ", desc)
        for tok in marca.split():
            if len(tok) > 2:
                desc = re.sub(rf"\b{re.escape(tok)}\b", " ", desc)

    # Adjetivos “ruido”
    noise = r"\b(sin|con|light|zero|diet|clasico|clasica|premium|familiar|suave|intenso|grande|mediano|chico|mini)\b"
    desc = re.sub(noise, " ", desc)

    # Espacios
    desc = re.sub(r"\s+", " ", desc).strip()
    return desc


# Generar embeddings (texto -> vector)
def generar_embeddings(texts: list[str],
                       model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                       batch_size_encode: int = 64,
                       batch_size_concat: int = 2048,
                       normalize: bool = True) -> np.ndarray:

    model = SentenceTransformer(model_name)
    emb_chunks = []
    for i in range(0, len(texts), batch_size_concat):
        print(f"Paso {i}/{len(texts)}/{batch_size_concat}")
        batch = texts[i:i + batch_size_concat]
        emb = model.encode(
            batch,
            batch_size=batch_size_encode,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=normalize
        )
        emb_chunks.append(emb)
        del emb
        gc.collect()
    embeddings = np.vstack(emb_chunks) if emb_chunks else np.empty((0, 384), dtype=np.float32)
    return embeddings


# Clusterizar embeddings (vector -> cluster_id)
def clusterizar_embeddings(embeddings: np.ndarray,
                           metodo: str = "kmeans",
                           n_clusters: int | None = None,
                           random_state: int = 42,
                           batch_size: int = 10_000,
                           **kwargs) -> np.ndarray:

    N = embeddings.shape[0]
    if N == 0:
        return np.array([], dtype=int)

    if n_clusters is None:
        n_clusters = int(min(5000, max(200, math.sqrt(N))))
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=random_state,
                          batch_size=batch_size, n_init="auto", **kwargs)
    labels = km.fit_predict(embeddings)
    return labels.astype(int)


# Construir producto_base (cluster_id -> etiqueta)
def construir_producto_base(df: pd.DataFrame,
                            col_texto_limpio: str = "desc_limpia",
                            col_cluster: str = "cluster_id",
                            top_k_tokens: int = 3) -> pd.DataFrame:

    STOPWORDS = set("""
    de del la las el los para por con en y o a x sin con al un una unos unas
    """.split())

    def top_tokens_for_cluster(sub: pd.Series) -> str:
        c = Counter()
        for s in sub.dropna():
            toks = [t for t in s.split() if t not in STOPWORDS and len(t) > 2]
            c.update(toks)
        if not c:
            return "misc"
        return " ".join([w for w, _ in c.most_common(top_k_tokens)])

    labels_df = (
        df.groupby(col_cluster)[col_texto_limpio]
          .apply(top_tokens_for_cluster)
          .rename("producto_base")
          .reset_index()
    )
    return df.merge(labels_df, on=col_cluster, how="left")


Creamos la nueva columna "desc_limpia":

In [ ]:
carrefour["desc_limpia"] = carrefour.apply(lambda r: clean_text_no_brand(r["productos_descripcion"], r.get("productos_marca")), axis=1)

Ahora utilizando la nueva columna "desc_limpia" generamos los embeddings, agrupamos en clusters y creamos la nueva columna "producto_base".

Como el proceso de cálculo de embeddings demora mucho, dividimos el flujo en dos: uno donde se hacen todos los cálculos y otro donde se utilizan los clusters ya calculados.

In [ ]:
EJECUTAR_PROCESO_COMPLETO = False

cluster_ids = None
if EJECUTAR_PROCESO_COMPLETO:
    embeddings = generar_embeddings(carrefour["desc_limpia"].tolist())
    np.save("./dataset/producto_embeddings.npy", embeddings)
    print("Total de filas embeddings:", embeddings.shape[0])
    print("Embeddings únicos:", np.unique(embeddings, axis=0).shape[0])
    # embeddings_cargado = np.load("./dataset/producto_embeddings.npy")

    cluster_ids = clusterizar_embeddings(embeddings, metodo="kmeans", n_clusters=None)
    np.save("./dataset/producto_cluster_ids.npy", cluster_ids)
else:
    cluster_ids = np.load("./dataset/producto_cluster_ids.npy")

print("Total de filas cluster_ids:", cluster_ids.shape[0])
print("Cluster_ids únicos:", np.unique(cluster_ids, axis=0).shape[0])

carrefour["cluster_id"] = cluster_ids
carrefour = construir_producto_base(carrefour, col_texto_limpio="desc_limpia", col_cluster="cluster_id", top_k_tokens=3)

In [ ]:
carrefour.head(10)

Como podemos ver, el proceso de cálculo del nombre de producto general da buenos resultados pero todavía necesita ser mejorado.

---

### Split del dataset

Por ahora sólo entre *training* y *test*.

In [ ]:
train_df, test_df = train_test_split(
    carrefour,
    test_size=0.2,
    shuffle=True
)

In [ ]:
print(train_df.shape, test_df.shape)

TODO: Falta split entre *features* y *target*

---

### Tratamiento de nulos

####  Eliminación de datos faltantes

Si hay demasiados valores faltantes o no se los puede imputar de manera confiable, podemos proceder a borrar las columnas sin afectar al dataset este es el caso de `productos_precio_unitario_promo2` y `productos_leyenda_promo2`, que no no tienen ningun valor.

El caso de `sucursales_observaciones`, `sucursales_barrio` y `sucursales_numero` no tiene sentido para nuestro modelo, asi que podemos borrarlas.

#### Imputaciones 

Nos quedan `productos_precio_unitario_promo1`, `productos_leyenda_promo1`. El texto de la promo podemos borrarlo, ya que no tiene mucho sentido en nuestro caso. Para el caso del precio unitario, lo que podemos hacer es asumir que ya que no tiene promo el precio es el precio unitario. Con esta imputación esta columna tiene cierto sentido para el modelo.


In [ ]:
# Condición: la columna 'precio_promocional' es nula
condicion_train = train_df['productos_precio_unitario_promo1'].isnull()
condicion_test = test_df['productos_precio_unitario_promo1'].isnull()

# Asignación: en las filas que cumplen la condición, actualiza el valor
train_df.loc[condicion_train, 'productos_precio_unitario_promo1'] = train_df['productos_precio_lista']
test_df.loc[condicion_test, 'productos_precio_unitario_promo1'] = test_df['productos_precio_lista']


---

### Tratamiento de outliers

Un dato se considera outlier si es < (Q1 - 1.5 * IQR)) o > (Q3 + 1.5 * IQR)

In [ ]:
# Métodos estadísticos para detectar outliers
datos_a_detectar_outliers = train_df[['productos_precio_lista', 'productos_precio_unitario_promo1']]

Q1 = datos_a_detectar_outliers.quantile(0.25)
Q3 = datos_a_detectar_outliers.quantile(0.75)

IQR = Q3 - Q1
outliers_iqr = (datos_a_detectar_outliers < (Q1 - 1.5 * IQR)) | (datos_a_detectar_outliers > (Q3 + 1.5 * IQR))

print("Outliers")
print(f"productos_precio_lista: <{Q1['productos_precio_lista'] - 1.5 * IQR['productos_precio_lista']:.02f} o >{Q3['productos_precio_lista'] + 1.5 * IQR['productos_precio_lista']:.02f}")
print(f"productos_precio_unitario_promo1: <{Q1['productos_precio_unitario_promo1'] - 1.5 * IQR['productos_precio_unitario_promo1']:.02f} o >{Q3['productos_precio_unitario_promo1'] + 1.5 * IQR['productos_precio_unitario_promo1']:.02f}")

Como pudimos ver en el trabajo practico anterior la cantidad de outliers grande, para minimizar el impacto de los outliers podemos aplicar logaritmo.


In [ ]:
# 2. Transformación logarítmica para reducir impacto de outliers
train_df['productos_precio_lista_log'] = np.log1p(train_df['productos_precio_lista'])
train_df['productos_precio_unitario_promo1_log'] = np.log1p(train_df['productos_precio_unitario_promo1'])

test_df['productos_precio_lista_log'] = np.log1p(test_df['productos_precio_lista'])
test_df['productos_precio_unitario_promo1_log'] = np.log1p(test_df['productos_precio_unitario_promo1'])


train_df.describe()[['productos_precio_lista_log','productos_precio_lista', 'productos_precio_unitario_promo1_log', 'productos_precio_unitario_promo1']]


In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 6), sharex=True)

# Reseteamos el índice para que el eje x sea continuo
train_df_plot = train_df.reset_index(drop=True)


# Primer gráfico: datos originales
axes[0].plot(train_df_plot.index, train_df_plot['productos_precio_lista'], alpha=0.7, color='gray')
axes[0].set_ylabel('Precio lista (original)')
axes[0].set_title('Precios lista original vs. Precios lista transformación logarítmica')
axes[0].get_yaxis().get_major_formatter().set_scientific(False) 


# Segundo gráfico: datos imputados
axes[1].plot(train_df_plot.index, train_df_plot['productos_precio_lista_log'], alpha=0.7, color='salmon')
axes[1].set_ylabel('Precios lista (transformado)')
axes[1].set_xlabel('Número de observación')
axes[1].get_yaxis().get_major_formatter().set_scientific(False) 


plt.tight_layout()
plt.show()

---

### Discretización

Como el objectivo es predecir el precio y lo más adecuado son árboles de decisión tipo Gradient Boosting / CatBoost, no necesitamos discretizar numéricos. Estos modelos ya hacen splits en rangos automáticamente. Lo mejor es mantenerlos como continuos y así preservar información.

En un caso real deberíamos probar el modelo así y si no se obtienen buenos resultados se podría aplicar modelos líneales, donde sí discretizar puede ser útil.

---

### Codificación

El objetivo de la codificación es transformar las variables categóricas en representaciones numéricas que preserven la mayor cantidad posible de información y, al mismo tiempo, sean adecuadas para el tipo de modelo que vamos a entrenar.

Estrategias de codificación vistas en clase:

- One-Hot Encoding: adecuado para variables con cardinalidad baja.

- Label Encoding y Ordinal Encoding: útiles en modelos basados en árboles o cuando existe un orden natural en las categorías.

- Target Encoding y Frequency Encoding: recomendados para variables con alta cardinalidad.

- Hashing Encoding: eficiente para atributos con gran número de valores únicos.

- Cyclic Encoding: pensado para variables con naturaleza periódica (fechas, horas, días).

En base a estos criterios seleccionamos las siguientes variables categóricas a codificar:

- producto_base
- sucursales_provincia
- sucursales_localidad
- sucursales_tipo 

#### producto_base

In [ ]:
print(train_df['producto_base'])

#### sucursales_provincia

Esta variable contiene el nombre de la provincia donde se encuentra la Sucursal, su cardinalidad máxima posible será 24, conciderando Ciudad Autónoma de Buenos Aires identificada como provincia. Este valor de cardinalidad no superará 24, suponiendo que no se crearan nuevas provicincias, nunca se sabe.   

Si bien la cardinalidad no es alta y se puede aplicar One-hot-encoding sin problemas, preferimos usar label encodig porque es más compacta y el modelo que se usará está basado en árbolor, por lo que no afecta una posible interpretación de orden numérico.   

In [ ]:
#Creamos el codificador
le = LabelEncoder()

#Agregamos un anueva columna con los valores codificados
train_df['sucursales_provincia_le']=le.fit_transform(train_df['sucursales_provincia'])


print(
    train_df[['sucursales_provincia', 'sucursales_provincia_le']]
    .drop_duplicates()
    .sort_values(by='sucursales_provincia_le')
    .reset_index(drop=True)
)

---

#### sucursales_localidad

In [ ]:
train_df['sucursales_localidad'].nunique()

Las localidades tiene un cardinalidad muy alta para utilizar One-hot-encoding, por lo que utilizamos Label encoding

In [ ]:
train_df['sucursales_localidad_le']=le.fit_transform(train_df['sucursales_localidad'])

print(
    train_df[['sucursales_localidad', 'sucursales_localidad_le']]
    .drop_duplicates()
    .sort_values(by='sucursales_localidad_le')
    .reset_index(drop=True)
)

#### sucursales_tipo

Indica el tipo de sucursal, según la documentación del dataset, esta variable puede tomar lso siguientes valores

- **Hipermercado**: más de 15 cajas. 

- **Supermercado**: entre 4 y 15 cajas

- **Autoservicio**: entre 1 y 3 cajas

- **Tradicional**: Mostrador sin línea de caja. 

- **Web**

En nuestro dataset no aparecen todos losvalores posibles para esta variable

In [ ]:
print("Cantidad de tipos de Sucursales: ",train_df['sucursales_tipo'].nunique())

Vamos codificar esta variable utilizando One-hot-encoding, por su baja carnalidad y con la ventaja de ganar interpretabilidad en los datos. Vamos a considerar todos los valores posibles de la variable, según se describe en la documentación, ya que el modelo las tendrá en cuenta y podrá procesarlas en caso que en el futuro esos valores formen parte de la entrada. Por supuesto el modelo no los considerará para la predicción porque no formaron parte del entrenamiento.



In [ ]:
# Definimos todas las categorías posibles
tipos = ["Hipermercado", "Supermercado", "Autoservicio", "Tradicional", "Web"]

# Convertimos la columna a Categorical con todas las categorías
train_df['sucursales_tipo'] = pd.Categorical(train_df['sucursales_tipo'], categories=tipos)

# Aplicamos One-Hot Encoding con todas las categorías
ohe_df = pd.get_dummies(train_df['sucursales_tipo'], prefix="tipo")

# Concatenamos al dataset original
train_df = pd.concat([train_df, ohe_df], axis=1)

print(train_df.columns) 

---

### Escalamiento

La única columna numérica es el precio del producto, que es nuestra variable objetivo. No aplicamos escalamiento al precio porque:

- Los modelos de regresión lineal, árboles de decisión, Random Forest, etc., trabajan bien con target numérico en su escala original.
- Si escalamos el target, el modelo aprende valores en esa escala transformada, y luego deberíamos deshacer el escalamiento para obtener el precio real.
- El precio tiene un ainterpretación directa que perderíamos al escalarlo.

---

### Balanceo

Balanceo puede ser necesario cuando el *target* es categórico: en nuestro caso no aplica.

---

### Dimensionalidad

---

## Conclusiones

TODO: Conclusiones

---